# Can we predict 12-month loan default?
Target: `defaulted`. Data: 8,000 loan applications, one row per `application_id` (unique), no date column.

Design decisions (checklist.md):
- **Split:** no time column exists and each entity has exactly one row, so there is no later-rows or whole-entity structure to respect. I use a stratified random 80/20 split, and the same structure (stratified K-fold) inside training. Row order is not assumed to be chronological.
- **Features:** all columns are recorded at application, i.e. before funding, so all are usable. `application_id` is an identifier and is dropped.
- **Candidates (named before any scoring):** (A) trivial baseline predicting the training prevalence; (B) logistic regression on scaled numeric + one-hot region, grid over C in {0.1, 1, 10}; (C) histogram gradient boosting, grid over `max_depth` in {2, 3, 5} and `learning_rate` in {0.03, 0.1}.
- **Selection metric:** ROC AUC in CV. **Threshold:** chosen on out-of-fold CV predictions of the chosen model to maximise F1. All preprocessing lives in a Pipeline, so it is fit within folds.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve, average_precision_score, confusion_matrix
SEED=0
df=pd.read_csv('data.csv')
y=df['defaulted']; X=df.drop(columns=['defaulted','application_id'])
print(df.shape, df.application_id.is_unique, 'prevalence', y.mean().round(4))

## 1. Hold out the test set (untouched until the final step)

In [ ]:
X_tr,X_te,y_tr,y_te=train_test_split(X,y,test_size=0.2,stratify=y,random_state=SEED)
print(len(X_tr),len(X_te))
X_tr.describe(include='all').T

## 2. Candidates and CV (training rows only)

In [ ]:
num=['income_k','debt_ratio','credit_lines','age','employment_years','loan_amount_k']
cat=['region']
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)
pre_lr=ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(handle_unknown='ignore'),cat)])
pre_gb=ColumnTransformer([('n','passthrough',num),('c',OneHotEncoder(handle_unknown='ignore',sparse_output=False),cat)])
cands={
 'A_baseline':(Pipeline([('m',DummyClassifier(strategy='prior'))]),{}),
 'B_logreg':(Pipeline([('p',pre_lr),('m',LogisticRegression(max_iter=1000))]),{'m__C':[0.1,1,10]}),
 'C_hgb':(Pipeline([('p',pre_gb),('m',HistGradientBoostingClassifier(max_iter=200,random_state=SEED))]),
          {'m__max_depth':[2,3,5],'m__learning_rate':[0.03,0.1]}),
}
res={}
for k,(pipe,grid) in cands.items():
    gs=GridSearchCV(pipe,grid or {'m__strategy':['prior']},scoring='roc_auc',cv=cv).fit(X_tr,y_tr)
    res[k]=gs; print(k,round(gs.best_score_,4),gs.best_params_ if grid else '')

## 3. Choice (made on CV only)
The choice is computed in the next cell from CV AUC and printed before the test set is touched. The grid-search score is slightly optimistic for the selected configuration; the test set gives the unbiased number.

In [ ]:
best=max(res,key=lambda k:res[k].best_score_)
model=res[best].best_estimator_
oof=cross_val_predict(model,X_tr,y_tr,cv=StratifiedKFold(5,shuffle=True,random_state=SEED+1),method='predict_proba')[:,1]
p,r,t=precision_recall_curve(y_tr,oof)
f1=2*p*r/np.maximum(p+r,1e-12)
thr=float(t[np.argmax(f1[:-1])])
print('CHOSEN:',best,res[best].best_params_,'| threshold',round(thr,4),'| CV OOF AUC',round(roc_auc_score(y_tr,oof),4),'| OOF F1',round(f1[:-1].max(),4))

## 4. Final: refit on all training rows, score the test set once

In [ ]:
model.fit(X_tr,y_tr)
proba=model.predict_proba(X_te)[:,1]; pred=(proba>=thr).astype(int)
print('TEST ROC AUC',round(roc_auc_score(y_te,proba),4))
print('TEST PR AUC ',round(average_precision_score(y_te,proba),4),'(prevalence',round(y_te.mean(),4),')')
print('TEST F1 @thr',round(f1_score(y_te,pred),4))
print(confusion_matrix(y_te,pred))
rng=np.random.default_rng(SEED); n=len(y_te); a=[]
for _ in range(1000):
    i=rng.integers(0,n,n)
    if y_te.values[i].min()!=y_te.values[i].max(): a.append(roc_auc_score(y_te.values[i],proba[i]))
print('AUC 95% bootstrap CI',np.percentile(a,[2.5,97.5]).round(3))

## Conclusion
Chosen on CV only: gradient boosting (depth 2, lr 0.03), CV AUC 0.776 vs 0.776 for logistic regression (a tie; baseline 0.5), threshold 0.166 from out-of-fold F1.
Test set, scored once: ROC AUC 0.766 (95% bootstrap CI 0.73-0.80), PR AUC 0.305 vs 10.4% prevalence, F1 0.37 at the threshold (80 of 167 defaults caught, 191 false alarms).

So defaults are moderately predictable (well above chance, far from reliable). Caveats: the split is random because no time or grouping information exists, so drift in future applications could lower performance; the threshold maximises F1 and should be re-chosen for the real cost of missed defaults vs. rejected good loans.